In [1]:
import os
import glob
import numpy as np
import pandas as pd

ROOT = r'D:\Bluestock work\bluestock-fintech-analysis1'

RAW_DIR = os.path.join(ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(ROOT, 'data', 'processed')
REPORTS_DIR = os.path.join(ROOT, 'reports')

print('Raw folder exists:', os.path.exists(RAW_DIR))

paths = glob.glob(os.path.join(RAW_DIR, '*02_nav_history*'))
print(paths)

Raw folder exists: True
['D:\\Bluestock work\\bluestock-fintech-analysis1\\data\\raw\\02_nav_history.csv']


In [2]:
print(os.listdir(RAW_DIR))

['02_nav_history.csv', '1788499980509-304c1255-08_investor_transactions.csv', '1788499982117-e3d6ab98-09_portfolio_holdings.csv', '1788499982615-f9647ab2-10_benchmark_indices.csv', '1788499983024-b042c300-01_fund_master.csv', '1788499984134-b0cbf625-03_aum_by_fund_house.csv', '1788499984405-d702a6c6-04_monthly_sip_inflows.csv', '1788499984721-4b860901-05_category_inflows.csv', '1788499985036-da4a0c4a-06_industry_folio_count.csv', '1788499985420-bb134abf-07_scheme_performance.csv', 'nav_118632_raw.csv', 'nav_119092_raw.csv', 'nav_119551_raw.csv', 'nav_120503_raw.csv', 'nav_120841_raw.csv', 'nav_125497_raw.csv']


In [3]:
nav = pd.read_csv(paths[0])
n_start = len(nav)

print(nav.shape)
print(nav.dtypes)
print(nav.head(10))

(46000, 3)
amfi_code      int64
date             str
nav          float64
dtype: object
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692
5     119551  2022-01-10  55.2835
6     119551  2022-01-11  56.0878
7     119551  2022-01-12  56.4978
8     119551  2022-01-13  56.2934
9     119551  2022-01-14  56.5926


In [4]:
log = []

def add_log(step, df):
    log.append({'step': step, 'rows': len(df)})

add_log('start', nav)

In [5]:
DATE_FORMAT = '%Y-%m-%d'

raw_dates = nav['date'].copy()
nav['date'] = pd.to_datetime(nav['date'], format=DATE_FORMAT, errors='coerce')

n_bad_dates = nav['date'].isnull().sum()
print('Dates missing or not readable:', n_bad_dates)
print(raw_dates[nav['date'].isnull()].head(10))

Dates missing or not readable: 0
Series([], Name: date, dtype: str)


In [6]:
bad_date_rows = nav[nav['date'].isnull()]
nav = nav.dropna(subset=['date'])
nav['date'] = nav['date'].dt.normalize()

add_log('dates parsed, bad dates removed', nav)

today = pd.Timestamp.today().normalize()
print('Dates in the future:', (nav['date'] > today).sum())

Dates in the future: 0


In [7]:
nav_missing_before = nav['nav'].isnull().sum()
nav['nav'] = pd.to_numeric(nav['nav'], errors='coerce')
n_non_numeric = nav['nav'].isnull().sum() - nav_missing_before

print('Missing NAV values in the raw file:', nav_missing_before)
print('NAV values that are not numbers:', n_non_numeric)


Missing NAV values in the raw file: 0
NAV values that are not numbers: 0


In [8]:
n_exact_dups = nav.duplicated().sum()
nav = nav.drop_duplicates()
add_log('exact duplicates removed', nav)

print('Exact duplicates removed:', n_exact_dups)

Exact duplicates removed: 0


In [11]:
key_cols = ['amfi_code', 'date']

n_conflicts = nav.duplicated(subset=key_cols).sum()
print('Same fund and date, different NAV:', n_conflicts)

nav[nav.duplicated(subset=key_cols, keep=False)].sort_values(key_cols).head(10)

Same fund and date, different NAV: 0


,amfi_code,date,nav


In [12]:
removed_conflicts = nav[nav.duplicated(subset=key_cols, keep='last')]
nav = nav.drop_duplicates(subset=key_cols, keep='last')
add_log('same fund and date: kept last row', nav)

SORTING

In [13]:
nav = nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)
nav.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [14]:
print('Funds:', nav['amfi_code'].nunique())
print('Rows on a Saturday or Sunday:', nav['date'].dt.dayofweek.isin([5, 6]).sum())
print('Missing NAV values now:', nav['nav'].isnull().sum())

rows_per_fund = nav.groupby('amfi_code').size()
print(rows_per_fund.describe())

Funds: 40
Rows on a Saturday or Sunday: 0
Missing NAV values now: 0
count      40.0
mean     1150.0
std         0.0
min      1150.0
25%      1150.0
50%      1150.0
75%      1150.0
max      1150.0
dtype: float64


In [15]:
bad_nav_mask = nav['nav'] <= 0
n_non_positive = bad_nav_mask.sum()
print('NAV zero or negative:', n_non_positive)

removed_bad_nav = nav[bad_nav_mask]
nav.loc[bad_nav_mask, 'nav'] = np.nan

NAV zero or negative: 0


In [16]:
MAX_FILL_DAYS = 7

daily = (
    nav.set_index('date')
       .groupby('amfi_code')['nav']
       .resample('D')
       .last()
       .reset_index()
)

daily['is_filled'] = daily['nav'].isnull()
n_calendar_rows = len(daily)
print('Rows after adding every calendar day:', n_calendar_rows)

Rows after adding every calendar day: 64320


In [17]:
daily['nav'] = daily.groupby('amfi_code')['nav'].ffill(limit=MAX_FILL_DAYS)

n_still_missing = daily['nav'].isnull().sum()
daily = daily.dropna(subset=['nav']).reset_index(drop=True)
n_filled = daily['is_filled'].sum()

add_log('calendar days added, NAV forward-filled', daily)

print('Rows filled with the last NAV:', n_filled)
print('Rows removed (gap longer than the limit, or no earlier NAV):', n_still_missing)

Rows filled with the last NAV: 18320
Rows removed (gap longer than the limit, or no earlier NAV): 0


In [18]:
assert daily['nav'].notnull().all(), 'Some NAV values are missing'
assert (daily['nav'] > 0).all(), 'Some NAV values are not positive'
assert not daily.duplicated(subset=['amfi_code', 'date']).any(), 'Repeated fund and date'
assert daily.equals(daily.sort_values(['amfi_code', 'date']).reset_index(drop=True)), 'Not sorted'
assert str(daily['date'].dtype).startswith('datetime64'), 'Date is not a datetime'

print('All checks passed')

All checks passed


In [19]:
daily.to_csv('../data/processed/nav_history_clean.csv', index=False)

rejected = pd.concat([
    bad_date_rows.assign(reason='date missing or not readable'),
    removed_conflicts.assign(reason='same fund and date, kept last row'),
    removed_bad_nav.assign(reason='NAV zero or negative'),
], ignore_index=True)

rejected.to_csv('../data/processed/nav_history_removed_rows.csv', index=False)

check = pd.read_csv('../data/processed/nav_history_clean.csv', parse_dates=['date'])
print(check.shape)
print(check.dtypes)

size_mb = os.path.getsize('../data/processed/nav_history_clean.csv') / 1_000_000
print('Clean file size (MB):', round(size_mb, 1))

(64320, 4)
amfi_code             int64
date         datetime64[us]
nav                 float64
is_filled              bool
dtype: object
Clean file size (MB): 2.1


In [20]:
lines = [
    '# NAV history cleaning summary',
    '',
    f'- Rows at start: {n_start}. Rows at end: {len(daily)}. The end has more rows because weekends and holidays were added.',
    f'- Missing NAV values in the raw file: {nav_missing_before}.',
    f'- Dates missing or not readable: {n_bad_dates} (rows removed).',
    f'- NAV values that were not numbers: {n_non_numeric}. NAV zero or negative: {n_non_positive}. Both were treated as missing and then filled.',
    f'- Exact duplicate rows removed: {n_exact_dups}.',
    f'- Same fund and date with different NAV: {n_conflicts} (kept the last row).',
    f'- Rows filled with the last NAV: {n_filled} (column is_filled = True). Longest fill: {MAX_FILL_DAYS} days.',
    f'- Rows removed because the gap was longer than {MAX_FILL_DAYS} days or no earlier NAV existed: {n_still_missing}.',
    f'- Funds: {daily["amfi_code"].nunique()}. Dates: {daily["date"].min().date()} to {daily["date"].max().date()}.',
    '- Final checks passed: no missing NAV, all NAV above 0, no repeated fund and date, sorted by amfi_code and date.',
    '',
    '## Assumptions to confirm with my manager',
    '- When a fund has two NAV values on one date, the last row is the correct one.',
    f'- A gap of up to {MAX_FILL_DAYS} days is a weekend or holiday. A longer gap is missing data.',
]

text = '\n'.join(lines)

with open('../reports/nav_cleaning_summary.md', 'w', encoding='utf-8') as f:
    f.write(text)

print(text)

# NAV history cleaning summary

- Rows at start: 46000. Rows at end: 64320. The end has more rows because weekends and holidays were added.
- Missing NAV values in the raw file: 0.
- Dates missing or not readable: 0 (rows removed).
- NAV values that were not numbers: 0. NAV zero or negative: 0. Both were treated as missing and then filled.
- Exact duplicate rows removed: 0.
- Same fund and date with different NAV: 0 (kept the last row).
- Rows filled with the last NAV: 18320 (column is_filled = True). Longest fill: 7 days.
- Rows removed because the gap was longer than 7 days or no earlier NAV existed: 0.
- Funds: 40. Dates: 2022-01-03 to 2026-05-29.
- Final checks passed: no missing NAV, all NAV above 0, no repeated fund and date, sorted by amfi_code and date.

## Assumptions to confirm with my manager
- When a fund has two NAV values on one date, the last row is the correct one.
- A gap of up to 7 days is a weekend or holiday. A longer gap is missing data.
